1- Load Data

In [10]:
from datasets import load_dataset
ds = load_dataset("TuwaiqAcademy/AISA-ArabicFC")
train = ds["train"]
dev = ds["dev"]
print(ds)


DatasetDict({
    train: Dataset({
        features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
        num_rows: 10550
    })
    dev: Dataset({
        features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
        num_rows: 545
    })
})


2- Inspect Data

In [11]:
print(train.features)
print(train[0].keys())
print(train[0]["tool_called"])
print(train[0]["dialect"])

{'text': Value('string'), 'requires_function': Value('bool'), 'tool_called': Value('string'), 'messages': List({'_think_for_train': Value('string'), 'content': Value('string'), 'role': Value('string'), 'think': Value('string'), 'tool_calls': List({'function': {'arguments': {'amount': Value('float64'), 'bank_name': Value('string'), 'category': Value('string'), 'check_in': Value('string'), 'check_out': Value('string'), 'city': Value('string'), 'country': Value('string'), 'currency': Value('string'), 'date': Value('string'), 'days': Value('float64'), 'departure_city': Value('string'), 'departure_date': Value('string'), 'destination_country': Value('string'), 'doctor_name': Value('string'), 'from_currency': Value('string'), 'guests': Value('float64'), 'id_number': Value('string'), 'insurance_number': Value('string'), 'iqama_number': Value('string'), 'items': Value('string'), 'medication_name': Value('string'), 'nationality': Value('string'), 'num_persons': Value('float64'), 'passport_numbe

3- Filter Islamic tools

In [12]:
islamic_tools = {
    "get_qibla_direction",
    "calculate_zakat",
    "search_quran",
    "calculate_inheritance",
}

train_islamic = train.filter(lambda ex: ex["tool_called"] in islamic_tools)
dev_islamic = dev.filter(lambda ex: ex["tool_called"] in islamic_tools)

print("Islamic train size:", len(train_islamic))
print("Islamic dev size:", len(dev_islamic))

Islamic train size: 1525
Islamic dev size: 73


4- Save locally

In [13]:
train_islamic.save_to_disk("aisa_islamic_train")
dev_islamic.save_to_disk("aisa_islamic_dev")

Saving the dataset (1/1 shards): 100%|██████████| 73/73 [00:00<00:00, 480.33 examples/s]


5- Read back

In [14]:
from datasets import load_from_disk
train_islamic = load_from_disk("aisa_islamic_train")
dev_islamic = load_from_disk("aisa_islamic_dev")
print(train_islamic, dev_islamic)


Dataset({
    features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
    num_rows: 1525
}) Dataset({
    features: ['text', 'requires_function', 'tool_called', 'messages', 'tools', 'tools_sampled', 'negative_category', 'dialect'],
    num_rows: 73
})


6- Basic stats

In [15]:
from collections import Counter

print("Train tool counts:", Counter(train_islamic["tool_called"]))
print("Dev tool counts:", Counter(dev_islamic["tool_called"]))
print("Train dialect counts:", Counter(train_islamic["dialect"]))
print("Dev dialect counts:", Counter(dev_islamic["dialect"]))

Train tool counts: Counter({'get_qibla_direction': 519, 'search_quran': 506, 'calculate_zakat': 500})
Dev tool counts: Counter({'get_qibla_direction': 25, 'calculate_zakat': 24, 'search_quran': 24})
Train dialect counts: Counter({'msa': 854, 'levantine': 342, 'egyptian': 202, 'gulf': 114, 'maghrebi': 13})
Dev dialect counts: Counter({'msa': 44, 'levantine': 12, 'egyptian': 10, 'gulf': 5, 'maghrebi': 2})


EXAMPLES FOR US TO SEE


In [17]:
import json

def get_user_message(example):
    for msg in example["messages"]:
        if msg["role"] == "user":
            return msg["content"]
    return ""

def get_assistant_think(example):
    for msg in example["messages"]:
        if msg["role"] == "assistant":
            return msg.get("think", "")
    return ""

def get_assistant_tool_call(example):
    for msg in example["messages"]:
        if msg["role"] == "assistant":
            tool_calls = msg.get("tool_calls")
            if tool_calls:
                return tool_calls[0]
    return None

In [18]:
import json

islamic_tools = [
    "get_qibla_direction",
    "calculate_zakat",
    "search_quran",
]

for tool in islamic_tools:
    print("\n" + "#" * 100)
    print("TOOL:", tool)
    print("#" * 100)

    shown = 0
    for ex in train_islamic:
        if ex["tool_called"] == tool:
            print("\nUser:", get_user_message(ex))
            print("Dialect:", ex["dialect"])
            print("Think:", get_assistant_think(ex))
            print("Tool call:")
            print(json.dumps(get_assistant_tool_call(ex), ensure_ascii=False, indent=2))
            print("-" * 80)

            shown += 1
            if shown == 2:
                break


####################################################################################################
TOOL: get_qibla_direction
####################################################################################################

User: ما هو اتجاه القبلة من دبي؟
Dialect: msa
Think: يبحث المستخدم عن اتجاه القبلة من دبي. أداة get_qibla_direction هي الأنسب لأنها توفر معلومات دقيقة حول اتجاه القبلة بناءً على المدينة. المدخلات المتوقعة هي city=دبي.
Tool call:
{
  "function": {
    "arguments": {
      "amount": null,
      "bank_name": null,
      "category": null,
      "check_in": null,
      "check_out": null,
      "city": "دبي",
      "country": null,
      "currency": null,
      "date": null,
      "days": null,
      "departure_city": null,
      "departure_date": null,
      "destination_country": null,
      "doctor_name": null,
      "from_currency": null,
      "guests": null,
      "id_number": null,
      "insurance_number": null,
      "iqama_number": null,
      "items": nul

In [19]:
import pandas as pd

rows = []
for i in range(10):
    ex = train_islamic[i]
    rows.append({
        "dialect": ex["dialect"],
        "tool_called": ex["tool_called"],
        "user": get_user_message(ex),
        "think": get_assistant_think(ex)
    })

pd.DataFrame(rows)

,dialect,tool_called,user,think
0,msa,get_qibla_direction,ما هو اتجاه القبلة من دبي؟,يبحث المستخدم عن اتجاه القبلة من دبي. أداة get...
1,egyptian,get_qibla_direction,إزاي أعرف اتجاه القبلة في الأقصر,فهمت أنك تحتاج إلى معرفة اتجاه القبلة في مدينة...
2,msa,search_quran,أبحث عن آية تتحدث عن الصبر,يبدو أن المستخدم يبحث عن آية قرآنية تتعلق بموض...
3,msa,calculate_zakat,كم تبلغ زكاة المال إذا كان لدي 5000 جنيه مصري؟,يبدو أن المستخدم يرغب في معرفة مقدار زكاة الما...
4,msa,calculate_zakat,كم نسبة الزكاة على الذهب عيار ٢٤ اذا عندي ٥٠٠ ...,يبحث المستخدم عن حساب نسبة الزكاة على الذهب بن...
5,egyptian,search_quran,عايز أبحث عن كلمة 'الرحمن' في القرآن,"فهمت أنك ترغب في البحث عن كلمة ""الرحمن"" في الق..."
6,msa,search_quran,ابحث لي عن آية الكرسي في القرآن,يبدو أن المستخدم يرغب في العثور على آية الكرسي...
7,msa,get_qibla_direction,اتجاه القبلة في الاسكندرية إيه؟,فهمت أن المستخدم يبحث عن اتجاه القبلة في مدينة...
8,msa,get_qibla_direction,أريد معرفة اتجاه القبلة في جدة,يبدو أن المستخدم يرغب في معرفة اتجاه القبلة في...
9,msa,search_quran,دورلي في القرآن على الآية اللي بتقول 'وجعلنا م...,تحتاج إلى العثور على آية في القرآن تتعلق بجعل ...
